# Module 2: Govern + Test

> Part of the **Modular Workshops** series. Standalone, ~25 min.

Enabling a production ready agent with:

- **A governance layer** — in-stack redaction that keeps sensitive data (here, employee **salary**) out of LangSmith traces. We use the redaction controls from **PHI Best Practices**: **PII Middleware** (redacts salary in tool results before the model and the trace), **LangSmith SDK masking** (scrubs salary before trace egress when the model still needs the value), and **LangSmith Engine** as a detection backstop.

This module wires up these controls and enables testing through LangSmith Studio

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

print("LANGSMITH_API_KEY set:        ", bool(os.environ.get("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set:           ", bool(os.environ.get("OPENAI_API_KEY")))
print("TAVILY_API_KEY set:           ", bool(os.environ.get("TAVILY_API_KEY")))

---
# Part 1. Govern — Keep Salary Out of Traces

Our HR agent has a domain-specific sensitive field: **employee salary**, returned by the `lookup_employee` tool. That value lands in **tool-result messages** and would be written to every LangSmith trace unless we redact it.

**PHI Best Practices** describes several redaction controls. Which one(s) you reach for depends on one question: **does the model need to see the salary to do its job?**

| # | Control | Where it runs | Catches salary in traces by… | Use when |
|---|---|---|---|---|
| **1** | **PII Middleware** | agent runtime, inside the loop | redacting tool results *before* the model and the trace see them | the model should **not** see raw salary |
| **2** | **LangSmith SDK masking** | your app, before trace egress | scrubbing salary out of run inputs/outputs on the way to LangSmith | the model **does** need salary (comp Q&A) but the trace must not store it |
| **3** | **LangSmith Engine** | LangChain-hosted, after storage | flagging salary that slipped past the prevention layers | defense-in-depth backstop |

Below we wire up the two prevention controls against the real `lookup_employee` salary field, then point at the detection backstop.

> All HR data here is **synthetic** (see `utils/hr_seed.py`). Salaries below are fake.

### 1.1 PII Middleware on tool results

`PIIMiddleware` runs *inside the agent loop*. For a domain-specific field like salary there's no built-in detector, so we register a **custom PII type** with a regex detector. The key knob — and the one the PDF calls out as "most commonly missed" — is **`apply_to_tool_results=True`**: salary enters via the `lookup_employee` tool message, not user input, so input-only redaction would miss it entirely.

- `strategy="redact"` → replaces each match with a `[REDACTED_SALARY]` placeholder
- `apply_to_tool_results=True` → scrubs tool outputs before they re-enter the model *and* before they hit the trace
- because it modifies agent state, the trace inherits the redacted version — one config protects both the LLM call and the trace

Reach for this control when the model should **not** see raw salary (e.g. routing, summarizing, or drafting a case where the exact number is irrelevant). If the agent actually needs to reason over the number, use SDK masking (§1.2) instead so the trace is scrubbed without blinding the model.

In [ ]:
import re
from langchain.agents.middleware import PIIMiddleware
from langchain.agents.middleware._redaction import PIIMatch
from deepagents import create_deep_agent
from agents.hr_agent import lookup_employee, open_hr_case
from utils.models import model

# Custom detector for salary figures in the tool output, e.g. 'salary: $184,500'
# or 'salary=184500'. Returns PIIMatch(start, end) spans the strategy redacts.
SALARY_RE = re.compile(r"(?i)(?:salary\D{0,8})?\$?\s?\d{2,3},?\d{3}(?:\.\d{2})?")

def detect_salary(content: str) -> list[PIIMatch]:
    return [
        PIIMatch(type="salary", value=m.group(), start=m.start(), end=m.end())
        for m in SALARY_RE.finditer(content)
    ]

salary_middleware = PIIMiddleware(
    "salary",
    detector=detect_salary,
    strategy="redact",
    apply_to_tool_results=True,   # the knob most teams miss — salary arrives via tool output
    apply_to_output=True,         # also scrub if the model echoes a number back
)

redacting_agent = create_deep_agent(
    model=model,
    tools=[lookup_employee, open_hr_case],
    system_prompt=(
        "You are an internal HR assistant. Use lookup_employee to answer questions. "
        "Salary figures are redacted before you see them — never guess a number."
    ),
    middleware=[salary_middleware],
)

out = redacting_agent.invoke({
    "messages": [{"role": "user", "content": "What team and location is employee 1001 in?"}]
})
print(out["messages"][-1].text)
print("\nOpen this run's trace in LangSmith: the lookup_employee tool message shows [REDACTED_SALARY].")

### 1.2 LangSmith SDK masking before trace egress

Sometimes the agent **does** need the real salary — "is this offer within band?", "what's the raise percentage?". Redacting it with middleware would break the task. The PDF's rule: *if the model needs to see the data, use SDK masking alone* — it scrubs the **trace** on the way out without touching the model call.

The `langsmith.Client` exposes the masking knobs. We pass an **`anonymizer`** — a function applied to every serialized run's inputs and outputs just before egress. It walks the payload and rewrites salary figures, so LangSmith stores `[REDACTED_SALARY]` while the live agent still reasoned over the true value.

- `anonymizer` takes precedence over `hide_inputs` / `hide_outputs` and covers both
- coverage equals your masking logic — you own the patterns (reuse `SALARY_RE` from §1.1)
- this is the **foundation control**: lowest lift, no infrastructure changes
- the anonymizer lives on the **`langsmith.Client`**, so the trace is only masked if the run exports through a client that has it — a LangChain/LangGraph agent traces through the **global cached client** (`LANGSMITH_TRACING=true`), so we install the anonymizer on *that* client, before any run, and the whole agent trace comes through masked

Below we prove the split directly: we run an agent **with no salary redaction** (so the model genuinely sees the real number), with masking installed on the tracing client, then show the two sides — **what the agent saw** (raw `$…`) vs. **what LangSmith stores** (`[REDACTED_SALARY]`).

In [ ]:
from langsmith import Client
import langsmith.run_trees as rt

def mask_salary(payload):
    """Recursively rewrite salary figures in any string within a run payload."""
    if isinstance(payload, dict):
        return {k: mask_salary(v) for k, v in payload.items()}
    if isinstance(payload, list):
        return [mask_salary(v) for v in payload]
    if isinstance(payload, str):
        return SALARY_RE.sub("[REDACTED_SALARY]", payload)
    return payload

# The anonymizer runs on serialized inputs AND outputs just before the trace is
# exported — the model call upstream is untouched.
masking_client = Client(anonymizer=mask_salary)

# KEY STEP: install the masking client as the GLOBAL tracing client, BEFORE any
# traced run. LangGraph's tracer resolves its client from rt.get_cached_client()
# (the global singleton) and ignores tracing_context(client=...). By swapping the
# global for our masking client, the ONE full agent trace that LANGSMITH_TRACING
# already emits is exported through mask_salary — no second tracer, no partial
# 'just a tool call' trace. (In a deployment, set the anonymizer on the client
# the app traces through instead of poking the global.)
rt._CLIENT = masking_client

# An agent with NO salary middleware: the model must see the real number to
# answer a comp question. SDK masking protects the trace WITHOUT blinding it.
comp_agent = create_deep_agent(
    model=model,
    tools=[lookup_employee, open_hr_case],
    system_prompt=(
        "You are an internal HR comp analyst. Use lookup_employee and get_comp_bands "
        "to answer questions about pay. You may reason over exact salary figures."
    ),
)

# Trace normally — env-var tracing now flows through the masking client, so the
# complete trace (user turn -> tool call -> model reasoning -> answer) appears in
# LangSmith with only the salary figures redacted.
result = comp_agent.invoke(
    {"messages": [{"role": "user", "content": "Is employee 1001's salary within their band?"}]}
)

def _text(content):
    """Flatten message content to a string (it can be a str or a list of blocks)."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
        )
    return str(content)

# --- (A) What the AGENT saw: the raw lookup_employee result (unredacted) ---
# A deep agent may run tools inside a spawned sub-agent, so the salary tool
# message isn't guaranteed to sit in the top-level result['messages']. We call
# the tool directly to get the exact string the model is handed — no middleware,
# so it's the real figure.
raw_tool_msg = lookup_employee.invoke({"employee_id": 1001})
print("AGENT SAW (raw tool result, what the model reasons over):")
print("  ", raw_tool_msg.replace(chr(10), " | "))

# --- (B) What LANGSMITH STORES: the SAME payload after the egress anonymizer ---
# This is exactly the transform the global tracing client now applies on export.
stored_tool_msg = masking_client._anonymizer({"content": raw_tool_msg})["content"]
print("\nLANGSMITH STORES (after egress masking):")
print("  ", stored_tool_msg.replace(chr(10), " | "))

# The agent's final answer still assessed the band — it saw the true number.
print("\nAGENT FINAL ANSWER:")
print("  ", _text(result["messages"][-1].content).replace(chr(10), " | "))

assert "$" in raw_tool_msg and any(c.isdigit() for c in raw_tool_msg), "agent should see a real figure"
assert "[REDACTED_SALARY]" in stored_tool_msg, "trace should be masked"
print("\nSame data, two views: the model reasoned over the real salary; the trace stores only the mask.")

Open this run in LangSmith (project `hr-agent-workshop-...`): you'll see the **full trace** — the user turn, the `lookup_employee` tool call, the model's reasoning, and the final answer — but every salary figure in it reads `[REDACTED_SALARY]`. The agent's answer still assessed the band because it reasoned over the true figure in memory; only the exported trace is scrubbed.

**Why install on the global client instead of adding a tracer?** LangGraph's tracer resolves its client from `rt.get_cached_client()` and ignores `tracing_context(client=...)`. If you *also* attach a `LangChainTracer(client=masking_client)` callback while `LANGSMITH_TRACING=true`, you get **two tracers**: the env-var one writing the full (unmasked) trace through the global client, and the callback one writing a second, partial trace — which is why salary still showed up and the trace looked like 'just a tool call'. Swapping the global client (`rt._CLIENT = masking_client`) keeps a single, complete, masked trace.

**In a deployment / app:** set the anonymizer on the client the app traces through at startup rather than poking the module global — same effect, no reliance on an internal name.

> `_anonymizer` / `_hide_run_inputs` are the client's internal egress hooks — we call `_anonymizer` here only to *show* the exact bytes LangSmith would store. In normal use you never call them yourself; the tracer applies the anonymizer automatically on export.

### 1.3 LangSmith Engine (detection backstop)

§1.1 and §1.2 are **prevention**. The PDF's detection control, **LangSmith Engine**, runs *after* storage: a LangChain-hosted deep agent that continuously monitors *stored* traces and, when configured to detect salary/PHI patterns, clusters the offenders into a named issue with severity, a timeline, and links to the evidence traces. It can propose online evaluators, pull offending traces into an offline eval dataset, and even draft a root-cause PR.

Engine runs **after** storage, so it won't stop salary from being written — it's a safety net for whatever slips past prevention, and the evidence trail for a continuous-monitoring compliance story. Wiring Engine up against this deployment is covered in **Module 5** (`utils/engine.py`).

**Combining the controls for this agent:** the model doesn't need raw salary for most HR Q&A, so **PII Middleware (§1.1) + SDK masking (§1.2)** stacked gives belt-and-suspenders trace protection; add **Engine** on top as the backstop.

---
# Part 2. Test — Test the (Now-Governed) Agent

The redaction controls are in place. Next, test the agent itself.

We run studio on the agent in `agents/deep_agent/` with the `langgraph` CLI. Because `agents/deep_agent/agent.py` imports `model` from `utils.models`, it picks up the module's model config automatically — no extra flags, no separate config.

## 2.1 Project structure

A testable LangGraph project is a directory with a `langgraph.json` config at the root that points at one or more graph objects. We already have one — `langgraph.json` at the workshop root registers the deep agent at `agents/deep_agent/agent.py`.

`dependencies: ["."]` tells the CLI to install this project — `pyproject.toml` at the config root — into the image.


In [ ]:
import os

agent_dir = str(project_root / "agents" / "deep_agent")
print("langgraph.json (workshop root)")
print("---")

for root, dirs, files in os.walk(agent_dir):
    # Skip __pycache__ for clarity
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


In [ ]:
# langgraph.json -- the test configuration
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

# AGENTS.md -- the agent's identity
agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


The agent itself lives in `agent.py`. The module-level `agent` variable is what gets tested — `langgraph.json` references it as `"deep_agent": "./agents/deep_agent/agent.py:agent"`.


## 2.3 Local development

Three CLI commands you'll use (all run from the workshop root, where `langgraph.json` lives):

```bash
# Validate langgraph.json (imports each graph, checks deps)
langgraph validate

# Run locally for development (Studio UI + hot reload)
langgraph dev --port 2024

```

`langgraph dev` opens the LangGraph Studio UI in your browser — useful to step through tool calls and approve HITL interrupts visually. By default it connects to `https://smith.langchain.com` for the Studio frontend and talks to your local server.

**Docs:** [LangGraph CLI reference](https://docs.langchain.com/langsmith/langgraph-cli)


## 2.4 Validate the config

`langgraph validate` imports each graph in `langgraph.json` and checks the config — without building Docker or uploading anything. Use it to catch config and import errors before testing.


In [ ]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
!cd "{project_root}" && langgraph validate


## 2.6 What you get with LangSmith Studio

Once spun up, your agent is reachable through 30+ endpoints — you build it once, the platform exposes it everywhere:

| Capability | What you can do |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call your agent as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads (via the platform's Store) |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger your agent on a schedule |


## 2.7 A code-based salary-leak evaluator

Prevention (§1.1–1.2) and detection (§1.3) keep salary out of traces, but how do you *prove* a run stayed clean — in CI, or as an online check on live traces? A **code-based evaluator**: deterministic, cheap, no LLM judge. The rule is simple — **if any salary figure survives in the output, the run failed the control.**

The evaluator reuses the same `SALARY_RE` the redaction layers use (if the pattern catches it on the way in, the same pattern catches it leaking on the way out) and scans *everything the trace stores*: the final response **and** every tool message in the trajectory — because §1.1 showed salary enters via tool output, that's exactly where a leak hides.

It follows the repo's evaluator contract — `(inputs, outputs, reference_outputs) -> {"key", "score", "comment"}` — so it drops straight into `evaluate(...)` offline or an online evaluator on a deployment. `score=1.0` means clean; `score=0.0` means a leak, and the comment names the offending spans so you can jump to the trace.

In [ ]:
from langchain_core.messages import BaseMessage


def _iter_text(value):
    """Yield every string buried in a response/trajectory payload.

    Handles the shapes an agent run can hand back: a plain string, a list of
    LangChain messages, dict-shaped messages, or nested structures.
    """
    if value is None:
        return
    if isinstance(value, str):
        yield value
    elif isinstance(value, BaseMessage):
        yield from _iter_text(value.content)
    elif isinstance(value, dict):
        for v in value.values():
            yield from _iter_text(v)
    elif isinstance(value, (list, tuple)):
        for v in value:
            yield from _iter_text(v)


def salary_leak_evaluator(inputs, outputs, reference_outputs=None):
    """Fail (score=0.0) if any salary figure survives in the agent's output.

    Scans the final response AND every tool message in the trajectory using the
    same SALARY_RE the redaction layers use — redacted spans read as
    '[REDACTED_SALARY]' and no longer match, so a hit means the control was
    bypassed. Deterministic and LLM-free: safe for CI and online evaluators.
    """
    # Pull together everything the trace would store for this run.
    haystacks = []
    haystacks.append(outputs.get("response"))      # final answer
    haystacks.append(outputs.get("trajectory"))    # tool names / messages
    haystacks.append(outputs.get("messages"))      # full message list, if present

    leaks = []
    for payload in haystacks:
        for text in _iter_text(payload):
            # Ignore already-redacted placeholders; only raw figures count.
            if "[REDACTED_SALARY]" in text:
                text = text.replace("[REDACTED_SALARY]", "")
            leaks.extend(m.group() for m in SALARY_RE.finditer(text))

    clean = not leaks
    comment = (
        "no salary figures found in output"
        if clean
        else f"LEAK: {len(leaks)} salary figure(s) in output -> {leaks[:5]}"
    )
    return {"key": "salary_leak", "score": float(clean), "comment": comment}

Now take it to the LangSmith platform **two ways**, so the scores show up where you need them:

1. **Offline experiment** (`evaluate`) — score governed vs. ungoverned agents over a small dataset using the `salary_leak_evaluator` above; shows up as an **Experiment** you can compare.
2. **Online code evaluator** — a managed automation rule on your tracing project that scores live production traces automatically. Same logic, rewritten for LangSmith's in-platform `run`-signature code evaluator.

#### Path 1 — Offline experiment via `evaluate()`

`evaluate()` runs a **target** over a **dataset** and applies **evaluators**, writing an Experiment to LangSmith. We seed a tiny dataset of HR questions, then evaluate two targets — the governed agent (§1.1) and an ungoverned one — so the experiment comparison makes the leak obvious.

In [ ]:
from langsmith import Client, evaluate

ls_client = Client()  # uses LANGSMITH_API_KEY / endpoint from the environment

# --- A tiny dataset: a few HR questions whose answers would surface salary ---
DATASET_NAME = "module2-salary-leak"
questions = [
    "What is employee 1001's salary?",
    "Tell me about employee 1001 — role, location, and pay.",
    "Is employee 1001's salary within their band?",
]

# Create (or reuse) the dataset, then add examples. Inputs match the agent's
# invoke shape; there's no reference output — a leak check needs no gold answer.
if ls_client.has_dataset(dataset_name=DATASET_NAME):
    dataset = ls_client.read_dataset(dataset_name=DATASET_NAME)
else:
    dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description="HR questions used to check that salary never leaks into a trace.",
    )
    ls_client.create_examples(
        dataset_id=dataset.id,
        examples=[{"inputs": {"question": q}} for q in questions],
    )

n_examples = len(list(ls_client.list_examples(dataset_id=dataset.id)))
print(f"Dataset ready: {DATASET_NAME} ({n_examples} examples)")

In [ ]:
# The target runs an agent on a dataset example and returns the evaluator's
# `outputs` contract. We make a factory so we can score two agents.
def make_target(agent):
    def target(inputs: dict) -> dict:
        result = agent.invoke({"messages": [{"role": "user", "content": inputs["question"]}]})
        msgs = result["messages"]
        return {
            "response": msgs[-1].text,
            "messages": msgs,
            "trajectory": [getattr(m, "name", None) for m in msgs if getattr(m, "name", None)],
        }
    return target

# Reuse the governed agent from 1.1; build an ungoverned one for contrast.
ungoverned_agent = create_deep_agent(
    model=model,
    tools=[lookup_employee, open_hr_case],
    system_prompt="You are an internal HR assistant. Use lookup_employee to answer questions.",
)

# evaluate() writes one Experiment per call. salary_leak_evaluator plugs straight
# in — its (inputs, outputs, reference_outputs) signature is what evaluate expects.
governed_exp = evaluate(
    make_target(redacting_agent),
    data=DATASET_NAME,
    evaluators=[salary_leak_evaluator],
    experiment_prefix="salary-leak-governed",
    client=ls_client,
)
ungoverned_exp = evaluate(
    make_target(ungoverned_agent),
    data=DATASET_NAME,
    evaluators=[salary_leak_evaluator],
    experiment_prefix="salary-leak-ungoverned",
    client=ls_client,
)

print("\nTwo experiments written to LangSmith. Open each dataset's Experiments tab:")
print("  governed   -> salary_leak should be 1.0 across the board")
print("  ungoverned -> salary_leak drops to 0.0 where the raw figure leaked")

#### Path 2 — A LangSmith **online code evaluator**

An online evaluator is a managed **automation rule** on a tracing project: LangSmith runs it automatically over sampled production runs and writes the score as feedback on each trace — no app code, no cron. For a deterministic check like salary leakage, use a **Code Evaluator** (not LLM-as-judge).

A code evaluator runs *inside LangSmith*, so it differs from the offline one:

- **Signature:** it takes a single `run` argument and reads `run["outputs"]` / `run["inputs"]` — not `(inputs, outputs, reference_outputs)`.
- **Return:** a feedback dict like `{"salary_leak": 1}` (the key becomes the feedback name).
- **Must be inline and self-contained:** no importing `SALARY_RE` or `agents.hr_agent` from this repo, and **no network**. Allowed: the standard library plus `numpy`/`pandas`/`jsonschema`/`scipy`/`sklearn`.

So we write a **standalone** version of the check. The cell below is runnable here only to *test* it against a real trace's data; the function body is what you paste into LangSmith.

In [ ]:
# ----------------------------------------------------------------------------
# PASTE THIS BODY into LangSmith:  Tracing project -> Evaluators tab ->
# + Evaluator -> Code Evaluator. Name it 'salary_leak'. Self-contained on
# purpose (no repo imports, no network) so it runs in the sandbox.
# ----------------------------------------------------------------------------
def perform_eval(run):
    """LangSmith online code evaluator: flag salary figures leaked into a run.

    Scans the run's serialized inputs AND outputs. Returns {'salary_leak': 1}
    when clean, {'salary_leak': 0} when a raw salary figure survived.
    """
    import json
    import re

    salary_re = re.compile(r"(?i)(?:salary\D{0,8})?\$?\s?\d{2,3},?\d{3}(?:\.\d{2})?")

    # Flatten the whole run payload to text; redacted spans no longer match.
    blob = json.dumps(run.get("inputs", {})) + json.dumps(run.get("outputs", {}))
    blob = blob.replace("[REDACTED_SALARY]", "")
    leaked = bool(salary_re.search(blob))
    return {"salary_leak": 0 if leaked else 1}

In [ ]:
# Test the evaluator the way LangSmith will call it: pass a `run`-shaped dict
# (inputs + outputs) built from a real traced run, before pasting it into the UI.
from langchain_core.tracers.context import collect_runs

question = {"messages": [{"role": "user", "content": "What is employee 1001's salary?"}]}

with collect_runs() as cb:
    leak_result = ungoverned_agent.invoke(question)      # leaks the raw figure
    clean_result = redacting_agent.invoke(question)      # redacted by 1.1 middleware

def as_run(result):
    """Shape an invoke result like the `run` dict a code evaluator receives."""
    msgs = result["messages"]
    return {
        "inputs": question,
        "outputs": {"messages": [{"content": getattr(m, "content", "")} for m in msgs]},
    }

print("ungoverned run ->", perform_eval(as_run(leak_result)))   # expect {'salary_leak': 0}
print("governed run   ->", perform_eval(as_run(clean_result)))  # expect {'salary_leak': 1}

assert perform_eval(as_run(leak_result)) == {"salary_leak": 0}
assert perform_eval(as_run(clean_result)) == {"salary_leak": 1}
print("\nTested OK. Paste perform_eval into a Code Evaluator in LangSmith to run it online.")

**Wire it up in LangSmith (one-time, in the UI):**

1. **Tracing** → open your project (`hr-agent-workshop-...`) → **Evaluators** tab → **+ Evaluator** → **Code Evaluator**.
2. Name it `salary_leak` and paste the `perform_eval` body above.
3. *(Optional)* Add a **filter** so it only scores relevant runs — e.g. runs that called `lookup_employee` — and set a **sampling rate** (e.g. `0.1`) to control cost.
4. Click **Test Code** against a recent run, then **Save**. It now runs automatically over newly sampled traces and attaches a `salary_leak` score to each — the deterministic, un-bypassable complement to the Engine backstop in §1.3.

> Online evaluators are configured per project in LangSmith, not created from the Python SDK. Keeping `perform_eval` here (tested) means the UI copy is exactly what you validated locally.

**Docs:** [Set up online code evaluators](https://docs.langchain.com/langsmith/online-evaluations-code)

**Where this plugs in.** `evaluate(...)` gates releases in CI (fail the build if `salary_leak` drops below 1.0 on the dataset); the **online code evaluator** scores live production traces automatically — the deterministic, un-bypassable complement to the Engine backstop in §1.3. (Module 3 goes deeper on offline + online evals.)

## Recap

| Pillar | What | How |
|---|---|---|
| **Govern** | Redact salary in tool results (PII Middleware) | `PIIMiddleware("salary", detector=..., apply_to_tool_results=True)` |
| **Govern** | Scrub salary before trace egress (SDK masking) | `Client(anonymizer=mask_salary)` |
| **Govern** | Flag salary that slipped past prevention (Engine) | LangSmith Engine, see Module 5 |
| **Govern** | Prove a run stayed clean (code-based eval) | offline `evaluate(...)` experiment + LangSmith online **code evaluator** on live traces |
| **Test** | Testable graph config | `langgraph.json` at repo root |
| **Test** | Agent identity + skills | `AGENTS.md`, `skills/` |
| **Test** | Test it | `langgraph dev` |

**The pattern:** redact sensitive data in-stack (before the trace, and before the model when it doesn't need it) + one test command. The agent now lives behind a managed server with salary kept out of its traces, and 30+ endpoints are available to call it.

**Next:** Module 3 — LangSmith (tracing, querying traces, offline + online evals, annotation queues).